# Using LangChain to Interact With AI API

**Author:** Shinin Varongchayakul

**Date:** 30 Jan 2026

## 1. API Key

In [34]:
# Load API key

# Import packages
from pathlib import Path
from dotenv import load_dotenv
import os

# Get .env file path
PROJECT_ROOT = Path.cwd().parents[2]
env_path = PROJECT_ROOT / ".env"

# Load variables from .env
load_dotenv(env_path, override=True)

# Get Gemini API key
gemini_api_key = os.getenv("GEMINI_API_KEY_01")

## 2. LLM

In [35]:
# Import package
from langchain_google_genai import ChatGoogleGenerativeAI

# Create model instance
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.5,
    api_key=gemini_api_key
)

## 3. Prompt Template

In [36]:
# Import package
from langchain_core.prompts import ChatPromptTemplate

# Define system prompt
system_prompt = """
You are an expert curator of mental models across science, philosophy, and applied reasoning.

Your task is to explain mental models clearly and accurately using a fixed schema.

If the origin of a model is unclear or debated, state that explicitly.

Do not invent historical sources. Be concise and concrete.
"""

# Define user prompt
user_prompt = "Explain the following mental model: {model_query}"

# Create prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        # System prompt
        ("system", system_prompt.strip()),

        # User prompt
        ("human", user_prompt)
    ]
)

In [57]:
# Inspect prompt
prompt.format_messages(model_query="First Principles Thinking")

[SystemMessage(content='You are an expert curator of mental models across science, philosophy, and applied reasoning.\n\nYour task is to explain mental models clearly and accurately using a fixed schema.\n\nIf the origin of a model is unclear or debated, state that explicitly.\n\nDo not invent historical sources. Be concise and concrete.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Explain the following mental model: First Principles Thinking', additional_kwargs={}, response_metadata={})]

## 4. Output Format

In [38]:
# Import package
from pydantic import BaseModel, Field
from typing import List, Literal

# Define output structure
class MentalModel(BaseModel):

    # Mental model name
    model_name: str = Field(description="The commonly accepted name of the mental model")

    # Source/origin
    origin: str = Field(
        description="Where the model comes from (person, book, field, or cultural origin)"
    )

    # Brief description
    description: str = Field(
        description="A brief explanation of what the mental model is and why it matters"
    )

    # Examples
    examples: List[str] = Field(
        description="Concrete real-world examples illustrating the mental model"
    )

    # Cognitive load required to learn this model
    cognitive_load: Literal["low", "medium", "high"] = Field(
        description="How mentally demanding this model is to learn and apply"
    )

    # Tags
    tags: List[str] = Field(
        description="Short tags such as decision-making, systems thinking, learning, philosophy"
    )

In [39]:
# Add output structure to LLM
llm_with_structured_output = llm.with_structured_output(MentalModel)

## 5. Chain

In [40]:
# Build chain
chain = prompt | llm_with_structured_output

## 6. Run

In [41]:
# Run query
result = chain.invoke(
    {
        "model_query": "Compound Interest"
    }
)

In [46]:
# Import package
import json

# Load result
result_dict = result.model_dump()

# Print
print(json.dumps(result_dict, indent=4))

{
    "model_name": "The Answer to the Ultimate Question (42)",
    "origin": "The Hitchhiker's Guide to the Galaxy by Douglas Adams",
    "description": "The model of '42' from Douglas Adams's 'The Hitchhiker's Guide to the Galaxy' serves as a humorous yet profound reminder that the 'answer' to a question is often meaningless or unhelpful if the question itself is not properly understood or even well-posed. It highlights the absurdity of seeking simple, universal answers to complex, ill-defined problems, and emphasizes the importance of formulating the right questions before seeking solutions. It encourages critical thinking about the relationship between questions and answers.",
    "examples": [
        "A company invests heavily in a 'solution' (e.g., a new software system) without clearly defining the problem it's meant to solve, leading to a '42' scenario where they have an answer but no meaningful question.",
        "Someone asks 'What's the secret to success?' expecting a sing

## 7. Loop

In [50]:
# Create list of mental models
mental_model_queries = [
    "First Principles Thinking",
    "Occam's Razor",
    "Confirmation Bias",
    "Map is Not the Territory",
    "42"
]

# Instantiate collector
query_collector = []

# Loop through mental models
for query in mental_model_queries:

    # Send to Gemini
    result = chain.invoke(
        {
            "model_query": query
        }
    )

    # Add result to collector
    query_collector.append(result.model_dump())

In [ ]:
# Instantiate counter
i = 1

# Loop through elements in collector
for result in query_collector:

    # Print result
    print(f"Query {i}:")
    print(json.dumps(result, indent=4))
    print("\n")

    # Add 1 to counter
    i += 1

Query 1:
{
    "model_name": "First Principles Thinking",
    "origin": "Ancient Greek philosophy (Aristotle), popularized in modern business by Elon Musk",
    "description": "First Principles Thinking is the practice of breaking down complex problems into their most basic, fundamental truths and then building up new solutions from that foundation. Instead of reasoning by analogy or relying on existing assumptions, it encourages questioning everything to uncover the irreducible elements. It matters because it fosters innovation, deeper understanding, and the ability to solve problems in novel ways.",
    "examples": [
        "Elon Musk's approach to building rockets: instead of accepting high market prices, he broke down a rocket into its raw material costs, then innovated on manufacturing processes to drastically reduce expenses.",
        "Designing a new product by asking 'What is the absolute core function this needs to perform?' rather than 'How can we make a better version of w